In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/indonesia_dataset.csv")

df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


 View all unique crop labels and the total count

In [3]:
unique_crops = df['label'].unique()
print(f"Total unique crops: {df['label'].nunique()}")
print("Crop list:", unique_crops)

selected_crops = ['rice', 'maize', 'coffee', 'mango', 'cotton']

df_filtered = df[df['label'].isin(selected_crops)]

Total unique crops: 22
Crop list: <StringArray>
[       'rice',       'maize',    'chickpea', 'kidneybeans',  'pigeonpeas',
   'mothbeans',    'mungbean',   'blackgram',      'lentil', 'pomegranate',
      'banana',       'mango',      'grapes',  'watermelon',   'muskmelon',
       'apple',      'orange',      'papaya',     'coconut',      'cotton',
        'jute',      'coffee']
Length: 22, dtype: str


We only need `rice`, `maize`, `mungbean`, `coconut`, `mango`, `coffee`

### Trained Model W/O cross validation

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

selected_crops = ['rice', 'maize', 'mungbean', 'coconut', 'mango', 'coffee']
df_filtered = df[df['label'].isin(selected_crops)]

X = df_filtered.drop('label', axis=1)
y = df_filtered['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Support Vector Machine": SVC(kernel='rbf', random_state=42),
    "Gaussian Naive Bayes": GaussianNB()
}

for name, model in models.items():
    print(f"========== {name} ==========")

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"Overall Accuracy: {accuracy * 100:.2f}%\n")

    print("Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("\n")

========== Random Forest ==========
Overall Accuracy: 100.00%

Classification Report:
              precision    recall  f1-score   support

     coconut       1.00      1.00      1.00        17
      coffee       1.00      1.00      1.00        24
       maize       1.00      1.00      1.00        20
       mango       1.00      1.00      1.00        17
    mungbean       1.00      1.00      1.00        19
        rice       1.00      1.00      1.00        23

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120



========== Gradient Boosting ==========
Overall Accuracy: 100.00%

Classification Report:
              precision    recall  f1-score   support

     coconut       1.00      1.00      1.00        17
      coffee       1.00      1.00      1.00        24
       maize       1.00      1.00      1.00        20
       mango       1.00      1.00      1.00        17
    mung

### Trained Model with cross validation

In [9]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

selected_crops = ['rice', 'maize', 'mungbean', 'coconut', 'mango', 'coffee']
df_filtered = df[df['label'].isin(selected_crops)]

X = df_filtered.drop('label', axis=1)
y = df_filtered['label']

pipelines = {
    "Random Forest": Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(random_state=42))
    ]),
    "Gradient Boosting": Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    "Support Vector Machine": Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', random_state=42))
    ]),
    "Gaussian Naive Bayes": Pipeline([
        ('scaler', StandardScaler()),
        ('model', GaussianNB())
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = []

for name, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline, X, y, cv=cv, 
        scoring=['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted'],
        return_train_score=False
    )
    
    results.append({
        "Model": name,
        "Mean Accuracy": f"{scores['test_accuracy'].mean() * 100:.2f}% (±{scores['test_accuracy'].std() * 100:.2f}%)",
        "Mean Precision": f"{scores['test_precision_weighted'].mean():.4f}",
        "Mean Recall": f"{scores['test_recall_weighted'].mean():.4f}",
        "Mean F1-Score": f"{scores['test_f1_weighted'].mean():.4f}"
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

                 Model    Mean Accuracy Mean Precision Mean Recall Mean F1-Score
         Random Forest 100.00% (±0.00%)         1.0000      1.0000        1.0000
     Gradient Boosting  99.83% (±0.33%)         0.9984      0.9983        0.9983
Support Vector Machine 100.00% (±0.00%)         1.0000      1.0000        1.0000
  Gaussian Naive Bayes 100.00% (±0.00%)         1.0000      1.0000        1.0000
